In [1]:
!pip install -q segmentation-models-pytorch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 4.1 MB/s eta 0:00:0000:01


In [2]:
import os
import json
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
from tqdm import tqdm
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from sklearn.model_selection import StratifiedKFold

import torch
import albumentations as A
import segmentation_models_pytorch as smp
from albumentations.pytorch import ToTensorV2

In [3]:
# ==========================================
# 1. КОНФИГУРАЦИЯ И НАСТРОЙКИ
# ==========================================
DATA_ROOT = Path(r"/kaggle/input/competitions/dl-lab-3-product-segmentation/train")
IMAGES_DIR = DATA_ROOT / "images"
MASKS_DIR = DATA_ROOT / "masks"
N_SPLITS = 5
SEED = 42

# ПУТИ К ВЕСАМ
WEIGHTS_DIR_CONF1 = Path(r"/kaggle/input/datasets/tkachenko1van/conf1-unetpp-effb4-fold2-finetuned")
WEIGHTS_DIR_CONF2 = Path(r"/kaggle/input/datasets/tkachenko1van/conf2-manet-mitb3-fold1-finetuned")
WEIGHTS_DIR_CONF3 = Path(r"/kaggle/input/datasets/tkachenko1van/conf3-deeplab-convnext-fold1-finetuned") 

IMG_SIZE = 320
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

val_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

In [4]:
# ==========================================
# 2. ЗАГРУЗКА МОДЕЛЕЙ
# ==========================================
conf1_models, conf2_models, conf3_models = [], [], []

for fold in range(1, 6):
    # Conf 1
    m1 = smp.UnetPlusPlus(encoder_name="efficientnet-b4", encoder_weights=None, in_channels=3, classes=1).to(DEVICE)
    m1.load_state_dict(torch.load(WEIGHTS_DIR_CONF1 / f"conf1_unetpp_effb4_fold{fold}_finetuned.pth", map_location=DEVICE))
    m1.eval()
    conf1_models.append(m1)

    # Conf 2
    m2 = smp.MAnet(encoder_name="mit_b3", encoder_weights=None, in_channels=3, classes=1).to(DEVICE)
    m2.load_state_dict(torch.load(WEIGHTS_DIR_CONF2 / f"conf2_manet_mitb3_fold{fold}_finetuned.pth", map_location=DEVICE))
    m2.eval()
    conf2_models.append(m2)

    # Conf 3
    m3 = smp.DeepLabV3Plus(encoder_name="tu-convnext_small", encoder_weights=None, in_channels=3, classes=1).to(DEVICE)
    m3.load_state_dict(torch.load(WEIGHTS_DIR_CONF3 / f"conf3_deeplab_convnext_fold{fold}_finetuned.pth", map_location=DEVICE))
    m3.eval()
    conf3_models.append(m3)
        
print(f"✅ Всего загружено моделей в ансамбль: {len(conf1_models)+len(conf2_models)+len(conf3_models)}")

✅ Всего загружено моделей в ансамбль: 15


In [5]:
# ==========================================
# ПОДГОТОВКА ДАННЫХ И ФУНКЦИИ
# ==========================================
def prepare_dataframe():
    data = []
    for mask_path in sorted(MASKS_DIR.glob("*.png")):
        img_path = IMAGES_DIR / (mask_path.stem + ".jpg")
        if img_path.exists():
            mask_preview = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
            area = np.sum(mask_preview > 0)
            data.append({"image_path": str(img_path), "mask_path": str(mask_path), "area": area})
    
    df = pd.DataFrame(data)
    # Защита от дубликатов площадей (duplicates='drop')
    df['area_bin'] = pd.qcut(df['area'], q=5, labels=False, duplicates='drop') 
    print(f"Найдено {len(df)} изображений.")
    return df

df = prepare_dataframe()

def get_model_probs(model, img_bgr):
    original_h, original_w = img_bgr.shape[:2]
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    augmented = val_transform(image=img_rgb)
    img_orig = augmented['image'].unsqueeze(0).to(DEVICE)
    
    img_h, img_v, img_hv = torch.flip(img_orig, [3]), torch.flip(img_orig, [2]), torch.flip(img_orig, [2, 3])
    
    with torch.amp.autocast('cuda'):
        p_orig = torch.sigmoid(model(img_orig))
        p_h = torch.flip(torch.sigmoid(model(img_h)), dims=[3])
        p_v = torch.flip(torch.sigmoid(model(img_v)), dims=[2])
        p_hv = torch.flip(torch.sigmoid(model(img_hv)), dims=[2, 3])
        
    probs = (p_orig + p_h + p_v + p_hv) / 4.0
    probs_np = probs[0, 0].cpu().numpy().astype(np.float16) 
    
    if probs_np.shape != (original_h, original_w):
        probs_np = cv2.resize(probs_np.astype(np.float32), (original_w, original_h), interpolation=cv2.INTER_LINEAR).astype(np.float16)
    return probs_np

Найдено 2000 изображений.


In [6]:
# =======================================================
# 3. КЭШИРОВАНИЕ ВЕРОЯТНОСТЕЙ (Один раз, займет минут 10-15)
# =======================================================
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
oof_data = [] 

with torch.inference_mode():
    for fold, (train_idx, val_idx) in enumerate(skf.split(df, df['area_bin'])):
        print(f"Обработка валидации Фолда {fold + 1}...")
        val_df = df.iloc[val_idx].reset_index(drop=True)
        
        # Индексы fold в списке идут от 0 до 4. Это идеально совпадает с логикой обучения.
        model_c1, model_c2, model_c3 = conf1_models[fold], conf2_models[fold], conf3_models[fold]
        
        for i in tqdm(range(len(val_df))):
            img_path = val_df.loc[i, 'image_path']
            mask_path = val_df.loc[i, 'mask_path']
            
            img_bgr = cv2.imread(str(img_path))
            true_mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
            true_mask = (true_mask > 0).astype(np.uint8)
            
            p1 = get_model_probs(model_c1, img_bgr)
            p2 = get_model_probs(model_c2, img_bgr)
            p3 = get_model_probs(model_c3, img_bgr)
            
            oof_data.append({
                'p1': p1, 'p2': p2, 'p3': p3, 'true_mask': true_mask
            })

Обработка валидации Фолда 1...


100%|██████████| 400/400 [02:24<00:00,  2.76it/s]


Обработка валидации Фолда 2...


100%|██████████| 400/400 [02:14<00:00,  2.98it/s]


Обработка валидации Фолда 3...


100%|██████████| 400/400 [02:13<00:00,  2.99it/s]


Обработка валидации Фолда 4...


100%|██████████| 400/400 [02:16<00:00,  2.93it/s]


Обработка валидации Фолда 5...


100%|██████████| 400/400 [02:16<00:00,  2.92it/s]


In [7]:
# =======================================================
# 4. ПОИСК ИДЕАЛЬНЫХ ПАРАМЕТРОВ (Nelder-Mead + Векторизация)
# =======================================================
def objective(params):
    w1, w2, w3, T = params
    
    # Защита оптимизатора от некорректных значений
    if w1 < 0 or w2 < 0 or w3 < 0:
        return 1.0 
    if T < 0.1 or T > 0.9:
        return 1.0
        
    total_w = w1 + w2 + w3 + 1e-7 # Защита от деления на ноль
    
    dices = []
    
    # Считаем Dice для каждой картинки в её оригинальном размере
    for item in oof_data:
        # Взвешиваем вероятности
        ens_probs = (item['p1'] * w1 + item['p2'] * w2 + item['p3'] * w3) / total_w
        
        # Бинаризация
        preds = (ens_probs > T).astype(np.float32)
        target = item['true_mask'].astype(np.float32)
        
        # Расчет Dice для одной картинки
        intersection = np.sum(preds * target)
        union = np.sum(preds) + np.sum(target)
        dice = (2.0 * intersection + 1e-6) / (union + 1e-6)
        
        dices.append(dice)
    
    # Минимизируем ошибку (1 - средний Macro Dice)
    return 1.0 - np.mean(dices)

# Стартовая точка: [Вес 1, Вес 2, Вес 3, Порог]
x0 = [1.0, 1.0, 1.0, 0.45] 

# Ограничения: веса от 0 до 10, порог от 0.20 до 0.80
bounds = [(0.0, 10.0), (0.0, 10.0), (0.0, 10.0), (0.2, 0.8)]

result = minimize(
    objective, 
    x0, 
    method='Nelder-Mead', 
    bounds=bounds,
    options={'maxiter': 300, 'disp': True}
)

best_w1, best_w2, best_w3, best_T = result.x
best_dice = 1.0 - result.fun 

# Нормализуем веса, чтобы их сумма равнялась 1 (Так они красивее и понятнее)
total_best_w = best_w1 + best_w2 + best_w3
norm_w1 = best_w1 / total_best_w
norm_w2 = best_w2 / total_best_w
norm_w3 = best_w3 / total_best_w

print(f"\n🏆 ИДЕАЛЬНЫЕ ПАРАМЕТРЫ НАЙДЕНЫ:")
print(f"Вес Conf1 (Unet++):  {norm_w1:.4f} (или {norm_w1*100:.1f}%)")
print(f"Вес Conf2 (MAnet):   {norm_w2:.4f} (или {norm_w2*100:.1f}%)")
print(f"Вес Conf3 (DeepLab): {norm_w3:.4f} (или {norm_w3*100:.1f}%)")
print(f"Оптимальный порог T: {best_T:.4f}")
print(f"Максимальный OOF Dice: {best_dice:.6f}")

Optimization terminated successfully.
         Current function value: 0.078144
         Iterations: 55
         Function evaluations: 118

🏆 ИДЕАЛЬНЫЕ ПАРАМЕТРЫ НАЙДЕНЫ:
Вес Conf1 (Unet++):  0.1751 (или 17.5%)
Вес Conf2 (MAnet):   0.4283 (или 42.8%)
Вес Conf3 (DeepLab): 0.3965 (или 39.7%)
Оптимальный порог T: 0.4118
Максимальный OOF Dice: 0.921856
